# EDA — Exploratorio para Machine Learning
**Dataset:** `alquileres_etiquetados.csv`  
Ejecuta cada celda con **Shift + Enter**

In [ ]:
# ── Instala las librerias necesarias en el entorno actual ─────────────────
# Solo necesitas ejecutar esta celda UNA VEZ.
# Despues reinicia el kernel: Kernel -> Restart & Run All
import subprocess, sys

paquetes = ['matplotlib', 'pandas', 'numpy']
for pkg in paquetes:
    try:
        __import__(pkg)
        print(f'[OK] {pkg} ya esta instalado')
    except ImportError:
        print(f'[...] Instalando {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg], check=True)
        print(f'[OK] {pkg} instalado')

print('\nListo. Si se instalo algun paquete, haz: Kernel -> Restart & Run All')

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({'figure.dpi': 90})
print('Librerias cargadas OK')

## 1. Carga y resumen del dataset

In [ ]:
CSV_PATH = Path(r'D:/Proyectos/BI/10clase/sistemas_de_recomendacion/src/modules/alquileres_etiquetados.csv')
df = pd.read_csv(CSV_PATH)

FEATURES = ['ingreso', 'veces_alquilada_pelicula', 'precio_promedio_genero']
LABELS   = ['ingreso_alto', 'pelicula_popular', 'genero_alto_valor']
ALIAS    = {
    'ingreso_alto'     : 'Ingreso Alto',
    'pelicula_popular' : 'Pelicula Popular',
    'genero_alto_valor': 'Genero Alto Valor'
}

print(f'{df.shape[0]:,} filas x {df.shape[1]} columnas')
df.head(10)

In [ ]:
# Estadisticas descriptivas basicas
df[FEATURES + LABELS].describe().round(3)

## 2. Scatter Plot — Relacion entre variables
Cada punto es un registro del dataset.  
El color indica si el ingreso es alto (rojo=1) o bajo (azul=0).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Scatter Plot - Relacion entre variables', fontsize=13)

pares = [
    ('ingreso', 'veces_alquilada_pelicula', 'Ingreso vs Veces alquilada'),
    ('ingreso', 'precio_promedio_genero',   'Ingreso vs Precio promedio genero'),
]

for ax, (xcol, ycol, titulo) in zip(axes, pares):
    for val, etiqueta, color in zip([0, 1], ['ingreso_bajo', 'ingreso_alto'], ['steelblue', 'tomato']):
        sub = df[df['ingreso_alto'] == val]
        ax.scatter(sub[xcol], sub[ycol], alpha=0.3, s=8, label=etiqueta, color=color)
    ax.set_xlabel(xcol)
    ax.set_ylabel(ycol)
    ax.set_title(titulo)
    ax.legend()

plt.tight_layout()
plt.show()

## 3. Matriz de Covarianza
Mide cuanto se mueven juntas dos variables (en sus unidades originales).

- Valor positivo: suben juntas
- Valor negativo: una sube cuando la otra baja
- Cercano a 0: poca relacion lineal

**Nota:** la diagonal siempre es la varianza de cada variable.

In [ ]:
cov_matrix = df[FEATURES].cov().round(3)

print('Matriz de Covarianza:')
print(cov_matrix.to_string())
print()

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cov_matrix.values, cmap='Blues')
plt.colorbar(im, ax=ax)

cols = cov_matrix.columns.tolist()
ax.set_xticks(range(len(cols)))
ax.set_yticks(range(len(cols)))
ax.set_xticklabels(cols, rotation=20, ha='right', fontsize=9)
ax.set_yticklabels(cols, fontsize=9)

for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, str(cov_matrix.values[i, j]),
                ha='center', va='center', fontsize=9)

ax.set_title('Matriz de Covarianza (features numericos)')
plt.tight_layout()
plt.show()

## 4. Matriz de Correlacion
Normaliza la covarianza al rango **-1 a +1**, por lo que es comparable entre variables.

- +1: correlacion positiva perfecta
- -1: correlacion negativa perfecta
-  0: sin relacion lineal

Para ML es importante detectar features muy correlacionadas (multicolinealidad).

In [ ]:
corr_matrix = df[FEATURES].corr().round(3)

print('Matriz de Correlacion:')
print(corr_matrix.to_string())
print()

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr_matrix.values, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

cols = corr_matrix.columns.tolist()
ax.set_xticks(range(len(cols)))
ax.set_yticks(range(len(cols)))
ax.set_xticklabels(cols, rotation=20, ha='right', fontsize=9)
ax.set_yticklabels(cols, fontsize=9)

for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, str(corr_matrix.values[i, j]),
                ha='center', va='center', fontsize=9)

ax.set_title('Matriz de Correlacion (features numericos)')
plt.tight_layout()
plt.show()

## 5. Estadistica Acumulativa (ECDF)
La ECDF muestra para cada valor X que proporcion del dataset tiene un valor menor o igual a X.

- Eje X: valor de la variable
- Eje Y: probabilidad acumulada (0 a 1)

Sirve para ver rangos, outliers y distribucion sin necesidad de elegir bins.

In [ ]:
def ecdf(data):
    """Calcula los puntos x, y de la ECDF."""
    x = np.sort(data)
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Estadistica Acumulativa (ECDF) - features numericos', fontsize=13)

for ax, col in zip(axes, FEATURES):
    data = df[col].dropna().values
    x, y = ecdf(data)
    ax.plot(x, y, marker='.', linestyle='None', markersize=3, alpha=0.4)

    p25, p50, p75 = np.percentile(data, [25, 50, 75])
    ax.axvline(p50, color='red',   lw=1.5, ls='--', label='Mediana=%.2f' % p50)
    ax.axvline(p25, color='green', lw=1.0, ls=':',  label='P25=%.2f'     % p25)
    ax.axvline(p75, color='green', lw=1.0, ls=':',  label='P75=%.2f'     % p75)

    ax.set_xlabel(col)
    ax.set_ylabel('Proporcion acumulada')
    ax.set_title(col)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 6. Estadisticas agrupadas por etiqueta
Resumen numerico de los features segun cada etiqueta de clasificacion.

In [ ]:
for label in LABELS:
    print('\n=== ' + ALIAS[label] + ' ===')
    display(df.groupby(label)[FEATURES].agg(['mean', 'std', 'min', 'max']).round(3))